# Phase 1 : Collecte des données

## Objectif
Explorer et collecter les données de votes à l'Assemblée nationale française.

## Sources de données potentielles
1. **API Assemblée nationale officielle** : https://data.assemblee-nationale.fr/
2. **NosDéputés.fr** : API communautaire (Regards Citoyens)
3. **Open Data Parlement** : data.gouv.fr

## Ce notebook va :
- Tester l'accès aux différentes sources
- Récupérer un échantillon de données
- Explorer la structure des données
- Identifier les informations disponibles sur les votes

In [ ]:
# Imports
import requests
import pandas as pd
import json
from bs4 import BeautifulSoup
from datetime import datetime
import time
from pathlib import Path

# Configuration
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Test de l'API NosDéputés.fr

NosDéputés.fr offre une API REST accessible et bien documentée.

In [ ]:
# Test de l'API NosDéputés.fr
BASE_URL_NOSDEPUTES = "https://www.nosdeputes.fr"

def test_nosdeputes_api():
    """
    Teste l'accès à l'API NosDéputés.fr
    """
    try:
        # Liste des scrutins (votes)
        response = requests.get(f"{BASE_URL_NOSDEPUTES}/synthese/data/json", timeout=10)
        
        if response.status_code == 200:
            print("✓ Connexion à NosDéputés.fr réussie")
            data = response.json()
            print(f"Clés disponibles : {list(data.keys())}")
            return data
        else:
            print(f"✗ Erreur : {response.status_code}")
            return None
    except Exception as e:
        print(f"✗ Exception : {e}")
        return None

# Tester
nosdeputes_data = test_nosdeputes_api()

## 2. Exploration des scrutins (votes)

Un scrutin = un vote sur un texte/amendement

In [ ]:
def get_recent_scrutins(limit=10):
    """
    Récupère les scrutins récents
    """
    try:
        # Endpoint pour les scrutins
        response = requests.get(
            f"{BASE_URL_NOSDEPUTES}/scrutins/data/json",
            timeout=10
        )
        
        if response.status_code == 200:
            data = response.json()
            scrutins = data.get('scrutins', [])
            print(f"✓ {len(scrutins)} scrutins récupérés")
            return scrutins[:limit]
        else:
            print(f"✗ Erreur : {response.status_code}")
            return []
    except Exception as e:
        print(f"✗ Exception : {e}")
        return []

# Récupérer un échantillon
scrutins_sample = get_recent_scrutins(limit=10)

# Explorer la structure
if scrutins_sample:
    print("\n=== Structure d'un scrutin ===")
    print(json.dumps(scrutins_sample[0], indent=2, ensure_ascii=False)[:1000] + "...")

## 3. Analyse d'un scrutin détaillé

Récupérer les détails d'un vote spécifique avec la répartition par groupe

In [ ]:
def get_scrutin_details(scrutin_num):
    """
    Récupère les détails d'un scrutin spécifique
    """
    try:
        response = requests.get(
            f"{BASE_URL_NOSDEPUTES}/scrutin/{scrutin_num}/json",
            timeout=10
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"✗ Erreur pour scrutin {scrutin_num}: {response.status_code}")
            return None
    except Exception as e:
        print(f"✗ Exception : {e}")
        return None

# Tester avec un scrutin récent
if scrutins_sample:
    # Extraire le numéro du premier scrutin
    first_scrutin = scrutins_sample[0]
    scrutin_num = first_scrutin.get('numero', None)
    
    if scrutin_num:
        print(f"\nRécupération des détails du scrutin n°{scrutin_num}...")
        scrutin_details = get_scrutin_details(scrutin_num)
        
        if scrutin_details:
            print("\n=== Informations du scrutin ===")
            scrutin_info = scrutin_details.get('scrutin', {})
            print(f"Titre: {scrutin_info.get('titre', 'N/A')}")
            print(f"Date: {scrutin_info.get('date', 'N/A')}")
            print(f"\nGroupes parlementaires:")
            
            # Analyser les votes par groupe
            groupes = scrutin_info.get('groupes', {})
            if groupes:
                for groupe_data in groupes.get('groupe', []):
                    nom = groupe_data.get('organe_libelle', 'N/A')
                    pour = groupe_data.get('pour', {}).get('nombre', 0)
                    contre = groupe_data.get('contre', {}).get('nombre', 0)
                    abstention = groupe_data.get('abstention', {}).get('nombre', 0)
                    print(f"  - {nom}: Pour={pour}, Contre={contre}, Abstention={abstention}")

## 4. Test API Assemblée Nationale officielle

L'API officielle peut fournir des données plus complètes et à jour

In [ ]:
# API officielle Assemblée Nationale
BASE_URL_AN = "https://data.assemblee-nationale.fr/api"

def test_assemblee_api():
    """
    Teste l'API officielle de l'Assemblée Nationale
    Documentation : https://data.assemblee-nationale.fr/travaux-parlementaires/scrutins
    """
    try:
        # Tester l'endpoint des scrutins de la 16ème législature (actuelle)
        legislature = 16
        response = requests.get(
            f"https://data.assemblee-nationale.fr/api/opendata/scrutins/{legislature}",
            timeout=10
        )
        
        if response.status_code == 200:
            print("✓ Connexion à l'API officielle réussie")
            data = response.json()
            print(f"Type de réponse : {type(data)}")
            if isinstance(data, dict):
                print(f"Clés disponibles : {list(data.keys())[:10]}")
            elif isinstance(data, list):
                print(f"Nombre d'éléments : {len(data)}")
                if data:
                    print(f"Premier élément : {list(data[0].keys()) if isinstance(data[0], dict) else type(data[0])}")
            return data
        else:
            print(f"✗ Erreur : {response.status_code}")
            return None
    except Exception as e:
        print(f"✗ Exception : {e}")
        return None

# Tester
an_data = test_assemblee_api()

## 5. Synthèse et choix de la source de données

Comparer les sources et choisir la meilleure pour notre projet

In [ ]:
print("=== Comparaison des sources ===")
print("\n1. NosDéputés.fr")
print("   + API simple et accessible")
print("   + Données agrégées par groupe")
print("   + Historique disponible")
print("   - Peut être moins à jour")

print("\n2. API Assemblée Nationale officielle")
print("   + Source officielle et complète")
print("   + Données les plus à jour")
print("   + Format structuré")
print("   - Documentation parfois complexe")

print("\n>>> DÉCISION : Utiliser les deux sources")
print("    - API officielle pour les données récentes (législature 16)")
print("    - NosDéputés.fr comme backup et validation")

## 6. Sauvegarde de l'échantillon

Sauvegarder les données collectées pour analyse

In [ ]:
# Sauvegarder les données
if scrutins_sample:
    sample_path = DATA_DIR / "scrutins_sample.json"
    with open(sample_path, 'w', encoding='utf-8') as f:
        json.dump(scrutins_sample, f, ensure_ascii=False, indent=2)
    print(f"✓ Échantillon sauvegardé : {sample_path}")

if an_data:
    an_path = DATA_DIR / "assemblee_nationale_sample.json"
    with open(an_path, 'w', encoding='utf-8') as f:
        # Limiter la taille si c'est une liste
        data_to_save = an_data[:10] if isinstance(an_data, list) else an_data
        json.dump(data_to_save, f, ensure_ascii=False, indent=2)
    print(f"✓ Données AN sauvegardées : {an_path}")

## Prochaines étapes

1. ✅ Sources de données identifiées et testées
2. ⏳ Collecte massive des scrutins de la législature 16
3. ⏳ Extraction des textes de loi associés
4. ⏳ Construction de la matrice votes × groupes parlementaires

**Notebook suivant** : `02_vote_matrix.ipynb`